In [ ]:
import os
import random
import time
import copy
import numpy as np
import pandas as pd
import tracemalloc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.ops import sigmoid_focal_loss
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, classification_report, roc_curve, roc_auc_score
from tqdm import tqdm
import glob
import joblib

# 1. Configuration de la Reproductibilité
SEED = 42

def set_seed(seed=42):
    """Fixe les graines aléatoires pour garantir la reproductibilité totale des résultats."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Pour des convolutions déterministes (peut ralentir légèrement mais essentiel pour la science)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[INFO] Global Seed set to {seed}")

# 2. Gestion de l'environnement et des dossiers
def setup_env():
    """Crée l'arborescence nécessaire si elle n'existe pas."""
    directories = ['datasets', 'models']
    for d in directories:
        os.makedirs(d, exist_ok=True)
        print(f"[INFO] Dossier vérifié/créé : {d}/")
    
    # Vérification du dossier Input (Juste un warning si absent, car c'est l'utilisateur qui fournit les données)
    if not os.path.exists('CIC-IDS-2017/'):
        print("[WARNING] Le dossier input 'CIC-IDS-2017/' est introuvable. Assurez-vous d'y placer vos CSV.")
    else:
        print("[INFO] Dossier input 'CIC-IDS-2017/' détecté.")

# 3. Détection Hardware
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Unité de calcul active : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"[INFO] GPU : {torch.cuda.get_device_name(0)}")

# Exécution du Setup
set_seed(SEED)
setup_env()

# Configuration Pandas pour meilleure lisibilité
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

In [ ]:
# --- LISTE DES COLONNES BRUTES ---
RAW_COLUMNS = [
    'Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 
    'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 
    'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 
    'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 
    'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 
    'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 
    'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 
    'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 
    'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 
    'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 
    'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 
    'CWE Flag Count', 'ECE Flag Count', 'Down/Up Ratio', 'Average Packet Size', 
    'Avg Fwd Segment Size', 'Avg Bwd Segment Size', 'Fwd Header Length.1', 
    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 
    'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Subflow Fwd Packets', 'Subflow Fwd Bytes', 
    'Subflow Bwd Packets', 'Subflow Bwd Bytes', 'Init_Win_bytes_forward', 
    'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean', 
    'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 
    'Idle Min', 'Label'
]

# 1. FEATURES LINUX
features_linux = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Total Length of Fwd Packets',
    'Total Length of Bwd Packets',
    'Fwd Packet Length Max',
    'Fwd Packet Length Min',
    'Fwd Packet Length Mean',
    'Fwd Packet Length Std',
    'Bwd Packet Length Max',
    'Bwd Packet Length Min',
    'Bwd Packet Length Mean',
    'Bwd Packet Length Std',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow IAT Max',
    'Flow IAT Min',
    'Fwd IAT Total',
    'Fwd IAT Mean',
    'Fwd IAT Std',
    'Fwd IAT Max',
    'Fwd IAT Min',
    'Bwd IAT Total',
    'Bwd IAT Mean',
    'Bwd IAT Std',
    'Bwd IAT Max',
    'Bwd IAT Min',
    'Fwd PSH Flags',
    'Bwd PSH Flags',
    'Fwd Header Length',
    'Bwd Header Length',
    'Fwd Packets/s',
    'Bwd Packets/s',
    'Min Packet Length',
    'Max Packet Length',
    'Packet Length Mean',
    'Packet Length Std',
    'Packet Length Variance',
    'FIN Flag Count',
    'SYN Flag Count',
    'RST Flag Count',
    'PSH Flag Count',
    'ACK Flag Count',
    'URG Flag Count',
    'ECE Flag Count',
    'Down/Up Ratio',
    'Average Packet Size',
    'Init_Win_bytes_forward',
    'Init_Win_bytes_backward',
    'act_data_pkt_fwd',
    'min_seg_size_forward',
    'Active Mean',
    'Active Std',
    'Active Max',
    'Active Min',
    'Idle Mean',
    'Idle Std',
    'Idle Max',
    'Idle Min'
]

# 2. FEATURES RASPBERRY PI
features_pi = [
    'Flow Duration',
    'Total Fwd Packets',
    'Total Backward Packets',
    'Total Length of Fwd Packets',
    'Total Length of Bwd Packets',
    'Fwd Packet Length Max',
    'Fwd Packet Length Min',
    'Bwd Packet Length Max',
    'Bwd Packet Length Min',
    'Flow Bytes/s',
    'Flow Packets/s',
    'Fwd Header Length',
    'Bwd Header Length',
    'Fwd Packets/s',
    'Bwd Packets/s',
    'Min Packet Length',
    'Max Packet Length',
    'FIN Flag Count',
    'SYN Flag Count',
    'PSH Flag Count',
    'ACK Flag Count',
    'Down/Up Ratio',
    'Average Packet Size',
    'Init_Win_bytes_forward',
    'Init_Win_bytes_backward',
    'act_data_pkt_fwd',
    'min_seg_size_forward'
]

print(f"[INFO] Features totales brutes : {len(RAW_COLUMNS)}")
print(f"[INFO] Features Linux Finales : {len(features_linux)}")
print(f"[INFO] Features Pi Finales    : {len(features_pi)}")

# Vérifs de sécurité
assert 'Label' not in features_linux
assert 'Label' not in features_pi
# Vérif doublons
assert len(features_linux) == len(set(features_linux)), "Doublons détectés dans la liste Linux"
assert len(features_pi) == len(set(features_pi)), "Doublons détectés dans la liste Pi"

In [ ]:
def load_and_process_data(input_dir='CIC-IDS-2017/', save_scalers=False):
    print(">>> DÉMARRAGE DU TRAITEMENT DES DONNÉES...")
    
    # 1. Chargement des CSV
    all_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not all_files:
        raise FileNotFoundError(f"Aucun fichier CSV trouvé dans {input_dir}")
    
    print(f"[INFO] {len(all_files)} fichiers détectés. Chargement en cours...")
    
    # Lecture optimisée (on force les types pour éviter les erreurs de mémoire sur les gros fichiers)
    df_list = []
    for f in all_files:
        try:
            # On lit le CSV. 'strip' sur les noms de colonnes est vital pour CIC-IDS-2017
            df_chunk = pd.read_csv(f, skipinitialspace=True) 
            df_list.append(df_chunk)
            print(f"   -> Chargé : {os.path.basename(f)} ({df_chunk.shape})")
        except Exception as e:
            print(f"   [ERREUR] Impossible de lire {f}: {e}")
            
    df = pd.concat(df_list, ignore_index=True)
    
    # Nettoyage des noms de colonnes (retrait des espaces avant/après)
    df.columns = df.columns.str.strip()
    print(f"[INFO] Dataset Brut Fusionné : {df.shape}")
    
    # Suppression des colonnes métadonnées (IPs, ID, Timestamp) si elles existent
    # Note : La liste fournie dans le prompt ne les contient pas, mais par sécurité :
    cols_to_drop = ['Flow ID', 'Source IP', 'Source Port', 'Destination IP', 'Timestamp', 'Destination Port', 'Fwd Header Length.1']
    
    existing_drop = [c for c in cols_to_drop if c in df.columns]
    if existing_drop:
        df.drop(columns=existing_drop, inplace=True)

    # 3. Nettoyage Mathématique (Inf, NaN)
    # Remplacement des infinis par NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    # Suppression des lignes contenant NaN
    initial_rows = len(df)
    df.dropna(inplace=True)
    print(f"[INFO] Lignes supprimées (NaN/Inf) : {initial_rows - len(df)}")

    # 4. Encodage du Label (Binaire)
    # Tout ce qui n'est pas 'BENIGN' devient 1 (Malicious)
    print("[INFO] Encodage des Labels...")
    df['Label'] = df['Label'].apply(lambda x: 0 if str(x).strip() == 'BENIGN' else 1)
    
    # Affichage distribution
    dist = df['Label'].value_counts(normalize=True)
    print(f"[STAT] Distribution : Benign={dist[0]:.2%}, Malicious={dist[1]:.2%}")

    # 5. Préparation des Datasets Spécifiques (Linux vs Pi)
    y = df['Label']
    
    # Filtrage des features (Intersection entre colonnes présentes et listes définies)
    # On s'assure que toutes les features demandées sont bien dans le DF
    feat_linux_ok = [f for f in features_linux if f in df.columns]
    feat_pi_ok = [f for f in features_pi if f in df.columns]
    
    X_linux = df[feat_linux_ok]
    X_pi = df[feat_pi_ok]
    
    # Suppression des colonnes à Variance Nulle (Constantes) sur Linux uniquement (Pi a des features fixées)
    # Pour Pi, on garde la structure fixe pour l'inférence temps réel, même si constante.
    # Pour Linux, on optimise.
    variance = X_linux.var()
    const_cols = variance[variance == 0].index
    if len(const_cols) > 0:
        print(f"[INFO] Suppression de {len(const_cols)} colonnes constantes (Variance 0) pour Linux.")
        X_linux = X_linux.drop(columns=const_cols)

    print(f"[INFO] Dimensions Finales -> Linux: {X_linux.shape}, Pi: {X_pi.shape}")
    print("\t-> Features Linux Finales :", X_linux.columns.tolist())
    dropped_linux = list(set(RAW_COLUMNS) - set(X_linux.columns.tolist()))
    dropped_pi = list(set(RAW_COLUMNS) - set(X_pi.columns.tolist()))
    print(f"\t>>> Features supprimées ({len(dropped_linux)}) :", dropped_linux)
    print("\t-> Features Pi Finales    :", X_pi.columns.tolist())
    print(f"\t>>> Features supprimées ({len(dropped_pi)}) :", dropped_pi)

    # 6. Split Train/Test & Scaling
    # On split AVANT de scaler pour éviter le data leakage
    # On retourne des dictionnaires pour tout garder organisé
    data_linux = {}
    data_pi = {}
    
    # --- TRAITEMENT LINUX ---
    print("--- Processing Linux Data ---")
    X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_linux, y, test_size=0.22, stratify=y, random_state=SEED)
    
    scaler_l = QuantileTransformer()
    X_train_l_scaled = scaler_l.fit_transform(X_train_l)
    X_test_l_scaled = scaler_l.transform(X_test_l)
    
    data_linux = {
        'X_train': X_train_l_scaled, 'y_train': y_train_l.values,
        'X_test': X_test_l_scaled,   'y_test': y_test_l.values,
        'features': X_linux.columns.tolist()
    }
    
    if save_scalers:
        joblib.dump(scaler_l, 'datasets/scaler_linux.pkl')
        print("[SAVE] scaler_linux.pkl sauvegardé.")

    # --- TRAITEMENT PI ---
    print("--- Processing Pi Data ---")
    X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(X_pi, y, test_size=0.22, stratify=y, random_state=SEED)
    
    scaler_p = QuantileTransformer()
    X_train_p_scaled = scaler_p.fit_transform(X_train_p)
    X_test_p_scaled = scaler_p.transform(X_test_p)
    
    data_pi = {
        'X_train': X_train_p_scaled, 'y_train': y_train_p.values,
        'X_test': X_test_p_scaled,   'y_test': y_test_p.values,
        'features': X_pi.columns.tolist()
    }
    
    if save_scalers:
        joblib.dump(scaler_p, 'datasets/scaler_pi.pkl')
        print("[SAVE] scaler_pi.pkl sauvegardé.")
        
    return data_linux, data_pi

# Exécution du chargement (mettez save_scalers=True pour la première exécution)
# NOTE : Cela peut prendre du temps selon la RAM disponible.
try:
    data_linux, data_pi = load_and_process_data(input_dir='CIC-IDS-2017/', save_scalers=True) # type: ignore
    print(">>> PRÉPARATION TERMINÉE AVEC SUCCÈS.")
except FileNotFoundError as e:
    print(f"[STOP] {e}")
    print("Veuillez placer le dataset dans le dossier CIC-IDS-2017/ avant de continuer.")

In [ ]:
class IDS_Dataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        # On garde y en float pour BCEWithLogitsLoss (Target doit être float)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1) 

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def get_dataloaders(data_dict, batch_size=256):
    """
    Crée les DataLoaders pour Train, et Test.
    Calcule aussi le poids de la classe positive pour la Loss.
    """
    # Création des Datasets
    train_ds = IDS_Dataset(data_dict['X_train'], data_dict['y_train'])
    test_ds = IDS_Dataset(data_dict['X_test'], data_dict['y_test'])
    
    # Création des Loaders
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True)
    
    # Calcul du pos_weight pour gérer le déséquilibre
    n_pos = data_dict['y_train'].sum()
    n_neg = len(data_dict['y_train']) - n_pos
    if n_pos == 0:
        pos_weight = torch.tensor([1.0])
    else:
        pos_weight = torch.tensor([n_neg / n_pos])
    
    print(f"[INFO] DataLoaders créés. Batch size: {batch_size}")
    print(f"[INFO] Class Imbalance Weight (pos_weight): {pos_weight.item():.2f}")
    
    return train_loader, test_loader, pos_weight

BATCH_SIZE = 4096

print("--- DataLoaders LINUX ---")
train_loader_l, test_loader_l, pos_weight_l = get_dataloaders(data_linux, batch_size=BATCH_SIZE)

print("\n--- DataLoaders PI ---")
train_loader_p, test_loader_p, pos_weight_p = get_dataloaders(data_pi, batch_size=BATCH_SIZE)

print(f"Nombre total de lignes chargées (RAM) : {len(data_linux['X_train']) + len(data_linux['X_test'])}")
print(f"Nombre de batches par époque (Train)  : {len(train_loader_l)}")
print(f"Lignes vues par époque (calculé)      : {len(train_loader_l) * BATCH_SIZE}")

In [ ]:
# ==========================================
# A. DÉFINITION DES ARCHITECTURES
# ==========================================

# 1. Modèle MLP (fully connected)
class MLP(nn.Module):
    def __init__(self, input_dim, width_factor = 1.0):
        super(MLP, self).__init__() 
        hidden_dim1 = int(128 * width_factor)
        hidden_dim2 = int(64 * width_factor)
        hidden_dim3 = int(32 * width_factor)
        hidden_dim4 = int(16 * width_factor)      
        self.net = nn.Sequential(
            # Bloc 1 : Expansion
            nn.Linear(in_features=input_dim, out_features=hidden_dim1),
            nn.BatchNorm1d(num_features=hidden_dim1),
            nn.GELU(),
            nn.Dropout(0.3),
            
            # Bloc 2 : Compression
            nn.Linear(in_features=hidden_dim1, out_features=hidden_dim2),
            nn.BatchNorm1d(num_features=hidden_dim2),
            nn.GELU(),
            nn.Dropout(0.3),
            
            # Bloc 3 : Raffinement
            nn.Linear(in_features=hidden_dim2, out_features=hidden_dim3),
            nn.BatchNorm1d(num_features=hidden_dim3),
            nn.GELU(),
            nn.Dropout(0.2),
            
            # Bloc 4 : Décision finale
            nn.Linear(in_features=hidden_dim3, out_features=hidden_dim4),
            nn.BatchNorm1d(num_features=hidden_dim4),
            nn.GELU(),
            nn.Dropout(0.1)
        )
        
        self.output = nn.Linear(in_features=hidden_dim4, out_features=1)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        
        elif isinstance(m, nn.BatchNorm1d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)
                    
    def forward(self, x):
        x = self.net(x)
        return self.output(x)

# 2. Modèle CNN (1D Conv)
class CNN_1D(nn.Module):
    def __init__(self, input_dim, width_factor = 1.0):
        super(CNN_1D, self).__init__()
        hidden_dim1 = int(32 * width_factor)
        hidden_dim2 = int(64 * width_factor)
        hidden_dim3 = int(128 * width_factor)
        # Bloc 1 : Extraction de bas niveau
        self.layer1 = nn.Sequential(
            nn.Conv1d(in_channels=1, out_channels=hidden_dim1, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(hidden_dim1),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.2)
        )
        
        # Bloc 2 : Extraction de moyen niveau
        self.layer2 = nn.Sequential(
            nn.Conv1d(in_channels=hidden_dim1, out_channels=hidden_dim2, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(hidden_dim2),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=2),
            nn.Dropout(0.3)
        )
        
        # Bloc 3 : Extraction de haut niveau
        self.layer3 = nn.Sequential(
            nn.Conv1d(in_channels=hidden_dim2, out_channels=hidden_dim3, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm1d(hidden_dim3),
            nn.GELU(),
            nn.Dropout(0.3)
        )
        
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        self.classifier = nn.Sequential(
            nn.Linear(in_features=hidden_dim3, out_features=hidden_dim2),
            nn.BatchNorm1d(hidden_dim2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(in_features=hidden_dim2, out_features=1)
        )
        
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Conv1d, nn.Linear)):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.BatchNorm1d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = x.unsqueeze(1)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x
    

# 3. Modèle Self-Attention (Transformer Encoder Block)
class Self_Attention(nn.Module):
    def __init__(self, input_dim, width_factor = 1.0, embed_dim=32, num_heads=4, num_layers=3):
        super(Self_Attention, self).__init__()
        
        self.seq_len = 16 if width_factor < 1.0 else 32
        self.embed_dim = int(embed_dim * width_factor)
        while self.embed_dim % num_heads != 0:
            self.embed_dim += 1
        self.hidden_dim = int(128 * width_factor)
        
        self.feature_projection = nn.Sequential(
            nn.Linear(in_features=input_dim, out_features=self.seq_len * self.embed_dim),
            nn.LayerNorm(self.seq_len * self.embed_dim),
            nn.GELU()
        )
        
        self.pos_encoder = nn.Parameter(torch.randn(1, self.seq_len, self.embed_dim) * 0.02)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=self.embed_dim, 
            nhead=num_heads, 
            dim_feedforward=self.embed_dim*4, 
            dropout=0.3, 
            activation='gelu', 
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        self.classifier = nn.Sequential(
            nn.Linear(in_features=self.embed_dim, out_features=self.hidden_dim),
            nn.BatchNorm1d(self.hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(in_features=self.hidden_dim, out_features=1)
        )
        
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
        
        elif isinstance(m, nn.MultiheadAttention):
            nn.init.xavier_uniform_(m.in_proj_weight)
            nn.init.xavier_uniform_(m.out_proj.weight)
            if m.in_proj_bias is not None:
                nn.init.constant_(m.in_proj_bias, 0)
            if m.out_proj.bias is not None:
                nn.init.constant_(m.out_proj.bias, 0)
                
        elif isinstance(m, (nn.LayerNorm, nn.BatchNorm1d)):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)

    def forward(self, x):
        batch_size = x.size(0)
        x = self.feature_projection(x).view(batch_size, self.seq_len, self.embed_dim)
        x = x + self.pos_encoder
        x = self.transformer_encoder(x)
        x = x.mean(dim=1) 
        return self.classifier(x)

# ==========================================
# B. MOTEUR D'ENTRAÎNEMENT (ENGINE)
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        return sigmoid_focal_loss(
            inputs, 
            targets, 
            alpha=self.alpha, 
            gamma=self.gamma, 
            reduction=self.reduction
        )
        
def train_model(
    model, train_loader, val_loader, 
    pos_weight, epochs=20, patience=4, 
    model_name="model", threshold=0.5):
    """
    Boucle d'entraînement générique.
    """
    model = model.to(DEVICE)
    pw = pos_weight.item() if isinstance(pos_weight, torch.Tensor) else pos_weight
    alpha_val = pw / (1 + pw)
    criterion = FocalLoss(alpha=alpha_val, gamma=2, reduction='mean').to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-3)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    
    best_val_f1 = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())
    patience_counter = 0
    
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_precision': [], 'val_recall': [], 'val_auc': []}
    
    print(f"\n[START] Entraînement : {model_name}")
    start_time = time.time()

    for epoch in range(epochs):
        # --- TRAIN ---
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} - Training"):
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * inputs.size(0)
        
        epoch_loss = running_loss / len(train_loader.dataset)
        
        # --- VALIDATION ---
        model.eval()
        val_loss = 0.0
        all_preds = []
        all_labels = []
        all_probs = []
        
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} - Validation"):
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                
                probs = torch.sigmoid(outputs)
                preds = (probs > threshold).float()
                
                all_probs.extend(probs.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_loss /= len(val_loader.dataset)
        val_f1 = f1_score(all_labels, all_preds, zero_division=0)
        val_prec = precision_score(all_labels, all_preds, zero_division=0)
        val_rec = recall_score(all_labels, all_preds, zero_division=0)
        try:
            val_auc = roc_auc_score(all_labels, all_probs)
        except ValueError:
            val_auc = 0.0
        
        scheduler.step(val_f1)
        
        history['train_loss'].append(epoch_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_precision'].append(val_prec)
        history['val_recall'].append(val_rec)
        history['val_auc'].append(val_auc)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_loss:.4f} | Val Loss: {val_loss:.4f} | F1: {val_f1:.4f} | Precision: {val_prec:.4f} | Recall: {val_rec:.4f} | AUC: {val_auc:.4f}")
        
        # --- EARLY STOPPING & SAVE ---
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
            torch.save(model.state_dict(), f"models/{model_name}.pth")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"[STOP] Early Stopping déclenché à l'époque {epoch+1}")
                break
                
    time_elapsed = time.time() - start_time
    print(f"[FIN] Durée: {time_elapsed:.0f}s | Best Val F1: {best_val_f1:.4f}")
    
    # Charger les meilleurs poids
    model.load_state_dict(best_model_wts)
    return model, history, time_elapsed

In [ ]:
results_log = []
confusion_matrices = {}

THRESHOLD = 0.5
EPOCHS = 50
PATIENCE = 7
# Configuration des expériences
# Format: (Platform, Architecture, Class, TrainL, TestL, PosWeight, InputDim)
experiments = [
    # --- LINUX (Serveur) ---
    ('Linux', 'MLP', MLP, train_loader_l, test_loader_l, pos_weight_l, len(data_linux['features']), 1.0),
    ('Linux', 'CNN', CNN_1D, train_loader_l, test_loader_l, pos_weight_l, len(data_linux['features']), 1.0),
    ('Linux', 'Attn', Self_Attention, train_loader_l, test_loader_l, pos_weight_l, len(data_linux['features']), 1.0),
    
    # --- RASPBERRY PI (Edge) ---
    ('Pi', 'MLP', MLP, train_loader_p, test_loader_p, pos_weight_p, len(data_pi['features']), 0.5),
    ('Pi', 'CNN', CNN_1D, train_loader_p, test_loader_p, pos_weight_p, len(data_pi['features']), 0.5),
    ('Pi', 'Attn', Self_Attention, train_loader_p, test_loader_p, pos_weight_p, len(data_pi['features']), 0.5),
]

print(f"🚀 DÉMARRAGE DU BENCHMARK COMPLET SUR {DEVICE}...")
print("="*80)

for platform, arch, model_cls, t_loader, v_loader, p_weight, in_dim, width_factor in experiments:
    model_name = f"Best_{platform}_{arch}"
    print(f"\n>>> Entraînement : {model_name} (Features: {in_dim})")
    
    model = model_cls(input_dim=in_dim, width_factor=width_factor)
    trained_model, history, train_time = train_model(
        model, t_loader, v_loader, p_weight, 
        epochs=EPOCHS, patience=PATIENCE, model_name=model_name, threshold=THRESHOLD
    )
    
    plt.figure(figsize=(12, 4))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Val Loss')
    plt.title(f'{model_name} - Loss Curve')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['val_f1'], label='Val F1-Score', color='green')
    plt.title(f'{model_name} - F1 Score')
    plt.xlabel('Epochs')
    plt.ylabel('F1')
    plt.legend()
    
    plt.tight_layout()
    plt.show()
    
    trained_model.eval()
    
    sample_input, _ = next(iter(v_loader))
    sample_input = sample_input.to(DEVICE)
    tracemalloc.start()
    start_inf = time.time()
    with torch.no_grad():
        for _ in range(100):
            _ = trained_model(sample_input)
    end_inf = time.time()
    current_mem, peak_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    inference_time_us = ((end_inf - start_inf) / (100 * sample_input.size(0))) * 1_000_000
    memory_usage_mb = peak_mem / (1024 * 1024)
    
    model_path = f"models/{model_name}.pth"
    size_kb = os.path.getsize(model_path) / 1024
    
    y_true = []
    y_pred = []
    y_probs = []
    
    with torch.no_grad():
        for inputs, labels in v_loader:
            inputs = inputs.to(DEVICE)
            outputs = trained_model(inputs)
            probs = torch.sigmoid(outputs)
            preds = (probs > THRESHOLD).cpu().numpy()
            y_true.extend(labels.numpy())
            y_pred.extend(preds)
            y_probs.extend(probs.cpu().numpy())
            
    f1 = f1_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    try:
        roc_auc = roc_auc_score(y_true, y_probs)
    except:
        roc_auc = 0.0
    fpr, tpr, _ = roc_curve(y_true, y_probs)
    
    print(classification_report(y_true, y_pred, target_names=['Benign', 'Malicious'], digits=4))
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    labels = [f"{val}\n({pct:.1f}%)" for val, pct in zip(cm.flatten(), cm_norm.flatten())]
    labels = np.asarray(labels).reshape(2,2)
    sns.heatmap(cm, annot=labels, fmt='', cmap='Blues', cbar=False, 
            xticklabels=['Benign', 'Malicious'], yticklabels=['Benign', 'Malicious'])
    plt.title(f"Confusion Matrix - {model_name}")
    plt.ylabel('Réel')
    plt.xlabel('Prédit')
    plt.show()
    
    results_log.append({
        'Platform': platform,
        'Arch': arch,
        'Accuracy': f"{acc:.4f}",
        'F1-Score': f"{f1:.4f}",
        'Precision': f"{prec:.4f}",
        'Recall': f"{rec:.4f}",
        'ROC-AUC': f"{roc_auc:.4f}",
        'FPR': (cm[0,1] / (cm[0,1] + cm[0,0])) * 100,
        'Inference': round(inference_time_us, 2),
        'Size': round(size_kb, 1),
        "Memory Peak": round(memory_usage_mb, 2),
        'Train Time': round(train_time, 1)
    })
    confusion_matrices[f"{model_name}_{platform}"] = cm
    
    print(f"-> Résultat : F1={f1:.4f} | Recall={rec:.4f} | AUC={roc_auc:.4f}")
    print("-" * 60)

In [ ]:
print("\n=== 🏆 TABLEAU COMPARATIF FINAL ===")
df_results = pd.DataFrame(results_log)
df_results['F1-Score'] = df_results['F1-Score'].astype(float)
df_results['Accuracy'] = df_results['Accuracy'].astype(float)
df_results['Precision'] = df_results['Precision'].astype(float)
df_results['Recall'] = df_results['Recall'].astype(float)
df_results['ROC-AUC'] = df_results['ROC-AUC'].astype(float)
df_results['FPR'] = df_results['FPR'].astype(float)
df_results['Inference'] = df_results['Inference'].astype(float)
df_results['Size'] = df_results['Size'].astype(float)
df_results['Memory Peak'] = df_results['Memory Peak'].astype(float)
df_results['Train Time'] = df_results['Train Time'].astype(float)

display_cols = [
    'Arch', 'Platform', 'Accuracy', 'Precision', 'Recall', 
    'F1-Score', 'ROC-AUC', 'FPR', 
    'Inference', 'Size', 'Memory Peak', 'Train Time'
]

def highlight_best(s):
    is_max = s == s.max()
    is_min = s == s.min()
    if s.name in ['FPR', 'Inference', 'Size', 'Memory Peak', 'Train Time']:
        return ['background-color: #d4edda' if v else '' for v in is_min]
    else:
        return ['background-color: #d4edda' if v else '' for v in is_max]

styled_df = df_results[display_cols].style.apply(highlight_best, subset=display_cols[2:])\
             .format("{:.4f}", subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC', 'FPR'])

display(styled_df)
df_results.to_csv("models/benchmark_results.csv", index=False)
df_melted = df_results.melt(id_vars=['Platform', 'Arch'], value_vars=['F1-Score', 'Recall'], var_name='Metric', value_name='Score')


df_metrics = df_results.melt(
    id_vars=['Arch', 'Platform'], 
    value_vars=['Accuracy', 'F1-Score', 'Precision', 'Recall', 'ROC-AUC', 'FPR'], 
    var_name='Metric', 
    value_name='Score'
)
sns.set_theme(style="whitegrid", font_scale=1.1)
g = sns.catplot(
    data=df_metrics, 
    kind="bar",
    x="Arch", 
    y="Score", 
    hue="Metric", 
    col="Platform",
    height=6, 
    aspect=1.2,
    palette="viridis",
    edgecolor="black",
    linewidth=1
)
g.figure.subplots_adjust(top=0.85)
g.figure.suptitle("Comparaison Détaillée", fontsize=16, fontweight='bold')
for ax in g.axes.flat:
    ax.set_ylim(0.0, 1.01) 
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', padding=3, fontsize=9, rotation=90)
    ax.set_ylabel("Score (0.0 - 1.0)")
    ax.set_xlabel("Architecture")
    ax.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()
    
sns.set_theme(style="whitegrid", context="talk")
plt.figure(figsize=(12, 8))
scatter = sns.scatterplot(
    data=df_results, 
    x='FPR', 
    y='Recall', 
    hue='Arch', 
    style='Platform', 
    size='F1-Score',
    sizes=(200, 600),
    palette='viridis',
    edgecolor='black',
    alpha=0.8
)
for i in range(df_results.shape[0]):
    plt.text(
        x=df_results.FPR[i], 
        y=df_results.Recall[i] + 0.002,
        s=f"{df_results.Arch[i]}", 
        fontdict=dict(color='black', size=10, weight='bold'),
        ha='center'
    )
plt.axvline(x=1.0, color='red', linestyle='--', linewidth=2, label="Limite Max FPR (1%)")
plt.axhline(y=0.95, color='green', linestyle='--', linewidth=2, label="Objectif Recall (95%)")
plt.title("Analyse de Fiabilité IDS : Taux de Détection vs Fausses Alarmes", fontsize=16, weight='bold', pad=20)
plt.xlabel("Faux Positifs (FPR %) [Moins c'est mieux] ➔", fontsize=12)
plt.ylabel("Taux de Détection (Recall) [Plus c'est mieux] ➔", fontsize=12)
fpr_max = df_results['FPR'].max()
recall_min = df_results['Recall'].min()
plt.xlim(-0.1, max(1.5, fpr_max * 1.2)) 
plt.ylim(max(0.8, recall_min - 0.05), 1.005)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
plt.text(0, 1.002, "⭐ ZONE IDÉALE", color='green', weight='bold', ha='left', va='bottom')
plt.tight_layout()
plt.show()

print("✅ Benchmarks terminés. Résultats sauvegardés dans 'models/benchmark_results.csv'.")
